In [1]:
import os
from datasets import load_dataset
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.trainers import WordLevelTrainer
from tokenizers.pre_tokenizers import Whitespace

def download_and_save_dataset(text_file_path="tinystories_dataset.txt"):
    print("1. Downloading TinyStories dataset from Hugging Face...")
    # This downloads the dataset into a Hugging Face cache
    dataset = load_dataset("roneneldan/TinyStories", split="train")
    
    print(f"2. Saving {len(dataset)} stories locally to {text_file_path}...")
    print("   (This might take a minute or two depending on your disk speed)")
    
    # Write everything to a local text file, one story per line (or block)
    with open(text_file_path, "w", encoding="utf-8") as f:
        for item in dataset:
            # We replace newlines in the story with spaces to keep one story per line,
            # which makes it much easier to load in your PyTorch DataLoader later.
            clean_story = item["text"].replace("\n", " ")
            f.write(clean_story + "\n")
            
    print(f"✅ Dataset successfully saved to {text_file_path}!")
    return text_file_path

def train_tokenizer_from_file(dataset_path, vocab_size=5000, save_path="tinystories_wordlevel.json"):
    print("\n3. Initializing Strict Word-Level Tokenizer...")
    tokenizer = Tokenizer(WordLevel(unk_token="[UNK]"))
    tokenizer.pre_tokenizer = Whitespace()
    
    trainer = WordLevelTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]", "[PAD]", "[BOS]", "[EOS]"],
        min_frequency=2, 
        show_progress=True
    )
    
    print(f"\n4. Training tokenizer on local file: {dataset_path}...")
    tokenizer.train([dataset_path], trainer=trainer)
    
    tokenizer.save(save_path)
    print(f"\n✅ Tokenizer successfully trained and saved to: {save_path}")
    print(f"Final Vocabulary Size: {tokenizer.get_vocab_size()}")

if __name__ == "__main__":
    local_text_file = "tinystories_dataset.txt"
    tokenizer_file = "tinystories_wordlevel.json"
    
    # Step 1: Download and save the text file
    if not os.path.exists(local_text_file):
        download_and_save_dataset(local_text_file)
    else:
        print(f"Dataset already exists at {local_text_file}, skipping download.")
        
    # Step 2: Train the tokenizer using that local file
    train_tokenizer_from_file(local_text_file, save_path=tokenizer_file)

c:\Users\MSI-1\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1. Downloading TinyStories dataset from Hugging Face...
2. Saving 2119719 stories locally to tinystories_dataset.txt...
   (This might take a minute or two depending on your disk speed)
✅ Dataset successfully saved to tinystories_dataset.txt!

3. Initializing Strict Word-Level Tokenizer...

4. Training tokenizer on local file: tinystories_dataset.txt...

✅ Tokenizer successfully trained and saved to: tinystories_wordlevel.json
Final Vocabulary Size: 5000
